# Project 2: retrieval, monoT5 reranking and Llama RAG

Run on an A40 or A30 Jupyter session, with this folder as the working directory. Run the cells in order.

In [ ]:
from utils import env

env.setup()

In [ ]:
import os
import shutil

import pandas as pd

from utils import config
import retrieval

In [ ]:
dev = retrieval.read_jsonl(retrieval.DATA_PATH)
print(len(dev), "dev questions")

## Step 1: top 15 BM25 passages for every dev question
Saved to `bm25_top15.json` in `DIR_OUTPUTS`. Questions already in the cache are skipped.

In [ ]:
searcher = retrieval.open_searcher(retrieval.INDEX)
cache = retrieval.build_cache(dev, searcher, threads=8)

## Step 2: score all 15 passages per question with monoT5
Saved to `t5_scores.jsonl` in `DIR_OUTPUTS`. Finished questions are skipped if you rerun this cell.

In [ ]:
retriever = retrieval.Retriever(cache=cache, searcher=searcher)
scorer = retrieval.T5Scorer()
retrieval.score_questions(dev, retriever, scorer)

## Step 3: threshold and cap sweep
Replays adaptive stopping on the saved scores, so no model runs here. `gold_in_bm25_topkeep` is the plain BM25 top-2 baseline.

In [ ]:
records = retrieval.read_jsonl(retrieval.SCORES_PATH)
rows = retrieval.sweep(records)
retrieval.write_sweep_csv(rows)
pd.DataFrame(rows)

## Step 4: Llama 3.1 8B Instruct reader
The model is gated: request access on its Hugging Face page, then log in once. The token is saved under `HF_HOME`, so later sessions can skip the login cell.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
import model
import rag

reader = model.LlamaReader()

## Step 5: EM and F1 for each retrieval setting
| Setting | Passages given to Llama |
|---|---|
| `closed_book` | none |
| `bm25_top2`, `bm25_top8` | BM25 top 2 or 8 |
| `monot5_top2`, `monot5_top8` | monoT5 rerank of the BM25 top 15, best 2 or 8 |
| `adaptive` | monoT5 in batches of 5 until a passage reaches 0.98 (at most 15), best 2 |

Selection reuses the Step 2 scores, so monoT5 doesn't run again. Predictions go to `llama/<setting>.jsonl` in `DIR_OUTPUTS` and are skipped on a rerun; the table is saved as `llama_eval.csv`. `answer_in_context` is the share of non-yes/no questions whose answer appears in the passages.

In [ ]:
eval_rows = rag.evaluate(records, reader)
pd.DataFrame(eval_rows).set_index("setting").round(2)

## Step 6: live pipeline
The same adaptive setting end to end on new input: BM25, then monoT5 scoring as the question comes in, then Llama. Times are per question.

In [ ]:
pipeline = rag.RAGPipeline(retriever, scorer, reader)
sample = dev[:5]
live = pipeline.answer_many(sample)
pd.DataFrame([{"question": q["question"], "gold": q["answer"], "answer": r["answer"],
               "passages": [p["title"] for p in r["passages"]], "scored": r["n_scored"],
               "seconds": r["retrieve_s"] + r["score_s"] + r["generate_s"]}
              for q, r in zip(sample, live)])

In [ ]:
archive = shutil.make_archive(os.path.join(config.ROOT, "project2_retrieval_results"), "zip",
                              os.path.dirname(config.DIR_OUTPUTS), os.path.basename(config.DIR_OUTPUTS))
print("Done. Download", archive)